### Step 1: Import python libraries and api key

In [8]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
import gradio as gr
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")

### Step 2: Set up Pushover

In [10]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

In [11]:
# test Pushover
import requests

def send_notification(message: str):
    payload={"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [12]:
#send_notification("Hello to myself from Pushover.")

### Step 3: Describe Pushover as an LLM tool

In [13]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a push notification to the user's phone via Pushover. User this to alert the user of time-sensitive issues.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pusherover to the list of tools for the LLM

In [14]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 5: Call the tool from an LLM

In [35]:
def handle_tool_call(tool_calls):
    tool_results=[]

    for tool_call in tool_calls:
        args=json.loads(tool_call.function.arguments)
        
        # Actually send the notification for the tool call
        send_notification(args["message"])

        tool_call_result={
            "role": "tool",
            "content": f"Notification sent: {args['message']}",
            "tool_call_id": tool_call.id
        }

        tool_results.append(tool_call_result)

    return tool_results

In [38]:
client=OpenAI()
messages=[
    {"role": "user", "content": "Please send me to separate notifications: 1) telling me what amazing progress I am making on the AI engineering training course. and 2) Hey there!."
    }
    ]

response=client.chat.completions.create(
    model="gpt-4o-mini", 
    messages=messages, 
    tools=tools,
)

# Check if the model wants to call a tool
message = response.choices[0].message

# check if model wants to call a tool
if message.tool_calls:
    tool_results=handle_tool_call(message.tool_calls) # whole list of tool calls on purpose
    messages.append(message)
    messages.extend(tool_results)  # changed from append() to extend() when we switched to multiple tool handling
 
    response=client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        #tools=tools #will add this in the future
    )
    
    message=response.choices[0].message
print(message.content)

1) Great job! You're making amazing progress on the AI engineering training course!

2) Hey there!


In [30]:
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_stoWNOSho9TVUkVoGq1wG5dO', function=Function(arguments='{"message":"Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!"}', name='send_notification'), type='function')])


In [32]:
if message.tool_calls:
    tool_call=message.tool_calls[0]
    import json
    args=json.loads(tool_call.function.arguments)

    # Actually send the notification
    send_notification(args["message"])
    print(f"Send notification: {args["message"]}")

else: 
    print(message.content)



Send notification: Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!
